# Great Expectations - Валидация музыкальных данных
Задание: Проверяем данные с помощью Great Expectations

In [1]:
import pandas as pd
import great_expectations as gx
import json
from datetime import datetime

In [2]:
df = pd.read_csv('dataset1.csv')
print(f'Размер данных: {df.shape}')

Размер данных: (114000, 21)


In [3]:
# Удаляем лишние колонки
columns_to_drop = [col for col in df.columns if col.startswith('Unnamed') or col == 'index']
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
print(f'Размер после очистки: {df.shape}')

Размер после очистки: (114000, 20)


In [4]:
# Получаем уникальные жанры
UNIQUE_GENRES = set(df['track_genre'].dropna().unique())
n_genres = len(UNIQUE_GENRES)
print(f'Уникальных жанров: {n_genres}')

Уникальных жанров: 114


In [5]:
# Создаём file context
import shutil
import os

# Очищаем старый context
context_dir = 'great_expectations'
if os.path.exists(context_dir):
    shutil.rmtree(context_dir)

context = gx.get_context(mode='file', context_root_dir=context_dir)
datasource = context.data_sources.add_pandas(name='music_data_source')
asset = datasource.add_dataframe_asset(name='music_data_asset')
batch_request = asset.build_batch_request(options={'dataframe': df})
print('Context создан')

Context создан


In [6]:
# Создаём Expectation Suite
expectation_suite = gx.ExpectationSuite(name='music_data_expectations')
expectation_suite = context.suites.add(expectation_suite)
print(f'Suite создан: {expectation_suite.name}')

Suite создан: music_data_expectations


### Добавление ожиданий в Suite

In [7]:
# Проверка обязательных колонок
required_columns = [
    'track_id', 'artists', 'album_name', 'track_name', 'popularity',
    'duration_ms', 'explicit', 'danceability', 'energy', 'key',
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature', 'track_genre'
]

for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnToExist(column=col)
    )
print(f'Добавлено проверок на наличие колонок: {len(required_columns)}')

Добавлено проверок на наличие колонок: 20


In [8]:
# Проверка на отсутствие пропусков
for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column=col)
    )
print(f'Добавлено проверок на пропуски: {len(required_columns)}')

Добавлено проверок на пропуски: 20


In [9]:
# Проверка track_id: длина строки строго 22 символа
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_id', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToEqual(column='track_id', value=22)
)

ExpectColumnValueLengthsToEqual(id='3997401c-612d-41cf-92ee-7119bbc2db73', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='track_id', mostly=1, row_condition=None, condition_parser=None, value=22.0)

In [10]:
# Проверка artists: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='artists', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='artists', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='01348a8f-5dda-4d87-b9c7-8747494db384', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='artists', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [11]:
# Проверка album_name: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='album_name', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='album_name', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='b0b44796-7381-45e0-b1c6-38f301929018', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='album_name', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [12]:
# Проверка track_name: длина строки от 2 до 512 символов
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_name', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column='track_name', min_value=2, max_value=512, strict_min=True, strict_max=True
    )
)

ExpectColumnValueLengthsToBeBetween(id='035846fa-aaa1-43cf-97ba-9dbaacaaa0a7', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='track_name', mostly=1, row_condition=None, condition_parser=None, min_value=2, max_value=512, strict_min=True, strict_max=True)

In [13]:
# Проверка popularity: int от 0 до 100
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='popularity', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='popularity', min_value=0, max_value=100, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='18323618-f68d-4f9b-b920-e48b7b2ca315', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='popularity', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=100.0, strict_min=False, strict_max=False)

In [14]:
# Проверка duration_ms: int от 0 (не включительно) до 5237760 (включительно)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='duration_ms', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='duration_ms', min_value=0, max_value=5237760, strict_min=True, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='e4668907-1161-497d-820c-26a2821e1956', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='duration_ms', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=5237760.0, strict_min=True, strict_max=False)

In [15]:
# Проверка explicit: bool
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='explicit', type_='bool')
)

ExpectColumnValuesToBeOfType(id='1ab34f54-003b-4699-b517-ca584ace2c3d', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='explicit', mostly=1, row_condition=None, condition_parser=None, type_='bool')

In [16]:
# Проверка danceability: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='danceability', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='danceability', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='372abce1-e7f4-4193-82f0-133b16f2ecfd', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='danceability', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [17]:
# Проверка energy: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='energy', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='energy', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='58a26592-4a63-480e-bb7c-457116e91f6c', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='energy', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [18]:
# Проверка key: int от 0 до 11
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='key', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='key', min_value=0, max_value=11, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='5c0177a3-2483-4298-90e6-ffe3fec2a1e1', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='key', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=11.0, strict_min=False, strict_max=False)

In [19]:
# Проверка loudness: float от -45 до 5
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='loudness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='loudness', min_value=-45, max_value=5, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='42489af9-0746-4e65-bfb3-647523532c52', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='loudness', mostly=1, row_condition=None, condition_parser=None, min_value=-45.0, max_value=5.0, strict_min=False, strict_max=False)

In [20]:
# Проверка mode: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='mode', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='mode', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='daa9af2e-8ad8-44c6-9635-a9845a202989', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='mode', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [21]:
# Проверка speechiness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='speechiness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='speechiness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='b339291e-6bb7-415e-bcd0-392429d0c9a7', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='speechiness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [22]:
# Проверка acousticness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='acousticness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='acousticness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='2d00565c-61b1-46f4-914b-9dee341c27a5', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='acousticness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [23]:
# Проверка instrumentalness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='instrumentalness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='instrumentalness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='531c9a7e-0f07-4f2c-95fa-368c829bc0f8', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='instrumentalness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [24]:
# Проверка liveness: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='liveness', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='liveness', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='d8f703cd-ffae-42f7-ad3c-426ab99eeeb7', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='liveness', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [25]:
# Проверка valence: float от 0 до 1
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='valence', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='valence', min_value=0, max_value=1, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='4823da58-85b0-43d8-a233-22bed32790cc', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='valence', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=1.0, strict_min=False, strict_max=False)

In [26]:
# Проверка tempo: float от 0 до 256
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='tempo', type_='float')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='tempo', min_value=0, max_value=256, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='9e3375a9-a75f-49e4-929a-5d10f75f7c04', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='tempo', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=256.0, strict_min=False, strict_max=False)

In [27]:
# Проверка time_signature: int от 0 до 5
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='time_signature', type_='int')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='time_signature', min_value=0, max_value=5, strict_min=False, strict_max=False
    )
)

ExpectColumnValuesToBeBetween(id='04e0df89-2ff8-4cb1-9b99-36ae4a623502', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='time_signature', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=5.0, strict_min=False, strict_max=False)

In [28]:
# Проверка track_genre: str, ограничение на 114 уникальных жанров
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_genre', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(column='track_genre', value_set=list(UNIQUE_GENRES))
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnUniqueValueCountToBeBetween(
        column='track_genre', min_value=n_genres, max_value=n_genres, strict_min=False, strict_max=False
    )
)
print(f'Добавлена проверка track_genre с {n_genres} уникальными значениями')

Добавлена проверка track_genre с 114 уникальными значениями


### Сохранение Expectation Suite в JSON

In [29]:
# Сохраняем suite в JSON файл
suite_json = expectation_suite.to_json_dict()
with open('music_data_expectations.json', 'w', encoding='utf-8') as f:
    json.dump(suite_json, f, indent=2, ensure_ascii=False)
print(f'Expectation Suite сохранён в music_data_expectations.json')
print(f'Количество ожиданий: {len(expectation_suite.expectations)}')

Expectation Suite сохранён в music_data_expectations.json
Количество ожиданий: 80


### Запуск проверки и получение результатов

In [30]:
# Запускаем валидацию через validator
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=expectation_suite
)

validation_result = validator.validate()
print(f'Проверка завершена: {validation_result.success}')

Calculating Metrics:   0%|          | 0/130 [00:00<?, ?it/s]

Проверка завершена: False


### Сохранение результатов проверки в JSON

In [31]:
results_dict = validation_result.to_json_dict()
with open('music_data_validation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_dict, f, indent=2, ensure_ascii=False)
print('Результаты проверки сохранены в music_data_validation_results.json')

Результаты проверки сохранены в music_data_validation_results.json


### Генерация HTML-отчёта (Data Docs)

In [32]:
# Очищаем старые Data Docs
data_docs_dir = os.path.join('great_expectations', 'uncommitted', 'data_docs')
if os.path.exists(data_docs_dir):
    shutil.rmtree(data_docs_dir)

# Строим Data Docs
context.build_data_docs()
print('HTML-отчёт сгенерирован')

# Путь к отчёту
index_path = os.path.abspath(os.path.join(data_docs_dir, 'local_site', 'index.html'))
print(f'Путь к отчёту: {index_path}')

HTML-отчёт сгенерирован
Путь к отчёту: C:\Projects\yp-sprint-5-practice-1\great_expectations\uncommitted\data_docs\local_site\index.html
